In [1]:
import gc
import os
import json
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoProcessor, VoxtralForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel, PeftConfig
import random
import wandb

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_NOTEBOOK_NAME"] = "qlora_evaluation.ipynb"
wandb.login()
compute_dtype = torch.bfloat16
model_id = "mistralai/Voxtral-Mini-3B-2507"
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_path = "./models/voxtral-glados-sft/final_adapters"

processor = AutoProcessor.from_pretrained(model_id)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=compute_dtype
)
base_model = VoxtralForConditionalGeneration.from_pretrained(
        model_id,
        attn_implementation="flash_attention_2",
        device_map="auto",
        low_cpu_mem_usage=True,
        quantization_config=bnb_config,
        dtype=compute_dtype
    )
peft_config = PeftConfig.from_pretrained(lora_path)
peft_config.init_lora_weights = False # avoid crash due to quantized initialization
model = PeftModel.from_pretrained(
    base_model,
    lora_path,
    config=peft_config,
    is_trainable=False
)
model.eval()
wandb.init(
    project=os.environ["WANDB_PROJECT"],
    job_type="evaluation",
    name="voxtral-glados-test-eval"
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vito/.netrc.
wandb: Currently logged in as: vitolus (vitolus-universit-ca-foscari-venezia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [2]:
def extract_json_payloads(text):
    if not isinstance(text, str):
        return []
    payloads = []
    # Find all non-overlapping { ... } blocks
    matches = re.finditer(r'(\{.*?\})', text, re.DOTALL)
    for match in matches:
        try:
            payloads.append(json.loads(match.group(1)))
        except json.JSONDecodeError:
            pass
    return payloads

def parse_generated_text(text):
    parts = text.split('\n\n', 1)
    json_section = parts[0]
    text_response = parts[1] if len(parts) > 1 else ""
    pred_payloads = extract_json_payloads(json_section)
    return pred_payloads, text_response, json_section

def calculate_slot_f1_multi(true_payloads, pred_payloads):
    def get_slot_set(payloads):
        slots = set()
        for p in payloads:
            if not p: # Handle empty dictionary {}
                continue
            # Associate the slot with its specific service to prevent cross-intent collisions
            srv = p.get("service", "unknown")
            for k, v in p.items():
                if k != "service":
                    slots.add((srv, k, str(v)))
        return slots
    true_set = get_slot_set(true_payloads)
    pred_set = get_slot_set(pred_payloads)
    # If both true and predicted require NO slots (Information Request {}), it's a perfect match.
    if not true_set and not pred_set:
        return 1.0
    tp = len(true_set.intersection(pred_set))
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    if (tp + fp) == 0 or (tp + fn) == 0:
        return 0.0
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if (precision + recall) == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def create_dynamic_instruction_fast(row, num_device_distractors=5, num_service_distractors=5):
    # Normalize double-escaped CSV quotes to standard JSON quotes instantly
    payload = str(row.get("Assistant_Payload", "")).replace('""', '"')
    req_devices, req_services = set(), set()
    # Extraction Phase
    if len(payload) > 2 and payload not in ["{}", "[]"]:
        p_lower = payload.lower()
        # O(N) generator to extract required devices
        req_devices.update(val for key, val in DEVICE_MAP.items() if "." in key and key in p_lower)
        # Highly simplified regex thanks to quote normalization
        req_services.update(re.findall(r'"service":\s*"([^"]+)"', payload, re.IGNORECASE))
    else:
        # Fallback for informative queries
        text = f"{row.get('User_Command', '')} {row.get('Target_GLaDOS_Response', '')}"
        req_devices.update(DEVICE_MAP.get(m.lower()) for m in DEVICE_REGEX.findall(text) if m.lower() in DEVICE_MAP)
    # Distractor Assembly Helper
    def mix(req_set, master_list, n):
        # set(master) - req_set automatically leaves only valid distractors
        pool = list(set(master_list) - req_set)
        final = list(req_set) + random.sample(pool, min(n, len(pool)))
        random.shuffle(final)
        return final
    # Prompt Construction
    devices = "\n".join(mix(req_devices, MASTER_DEVICES, num_device_distractors))
    services = ", ".join(mix(req_services, MASTER_SERVICES, num_service_distractors))
    return (
        f"You are GLaDOS, an AI assistant that controls the devices in a house. "
        f"Execute the spoken command, output the required JSON payload, and respond in character. "
        f"Complete the following task as instructed or answer the following question with the information provided only.\n"
        f"Services: {services}\n"
        f"Devices:\n{devices}\n"
    )

#### Load test dataset and evaluate

In [3]:
# Load your test dataset (using a CSV structure similar to your training script)
test_df = pd.read_csv("data/combined_multimodal_dataset_test.csv")
results = []
MASTER_SERVICES = "Services: climate.set_fan_mode(fan_mode), climate.set_humidity(humidity), climate.set_hvac_mode(), climate.set_preset_mode(), climate.set_temperature(temperature), climate.toggle(), climate.turn_off(), climate.turn_on(), cover.close_cover(), cover.open_cover(), cover.stop_cover(), cover.toggle(), fan.decrease_speed(), fan.increase_speed(), fan.toggle(), fan.turn_off(), fan.turn_on(), light.toggle(), light.turn_off(), light.turn_on(rgb_color,brightness), lock.lock(), lock.unlock(), media_player.media_next_track(), media_player.media_pause(), media_player.media_play(), media_player.media_play_pause(), media_player.media_previous_track(), media_player.media_stop(), media_player.toggle(), media_player.turn_off(), media_player.turn_on(), media_player.volume_down(), media_player.volume_mute(), media_player.volume_up(), switch.toggle(), switch.turn_off(), switch.turn_on(), timer.add_item(item), timer.cancel(), timer.pause(), timer.start(duration), vacuum.pause(), vacuum.return_to_base(), vacuum.start(), vacuum.stop()"
MASTER_DEVICES = [
    "climate.carrier_cor 'Carrier Cor Wi-Fi Thermostat' = auto;On High;24C;87%",
    "climate.ecobee_smart 'Ecobee SmartThermostat' = auto;Auto Low;78F;49%",
    "climate.emerson_sensi_touch 'Emerson Sensi Touch Wi-Fi Thermostat' = heat;Auto High;16C;68%",
    "cover.back_window 'Back Window Blinds' = closed",
    "cover.basement 'Basement Blinds' = closed",
    "cover.bathroom 'Bathroom Blinds' = open",
    "fan.attic 'Attic' = off",
    "fan.attic_ventilation 'Attic ventilation fan' = off",
    "fan.back_porch 'Back Porch Fan' = on",
    "light.above_dining_table 'Dining Table Light' = on;indianred (202, 92, 102);42%",
    "light.aquarium 'Aquarium Light' = off",
    "light.attic 'Attic Light' = off",
    "lock.attic 'Attic Door Lock' = unlocked",
    "lock.attic_door 'Attic Door' = unlocked",
    "lock.back_door 'Backyard entry lock' = unlocked",
    "media_player.apple_tv 'Apple TV media player' = off",
    "media_player.apple_tv_4k 'Apple 4K Streaming Device' = on;Martin: The Complete Five Seasons;vol=0.2",
    "media_player.attic_vinyl_turntable 'Attic Vinyl Record Player' = off",
    "switch.attic_lights 'Attic Lights Switch' = off",
    "switch.balcony_lighting 'Balcony lighting control' = on",
    "switch.balcony_mood_lighting 'Balcony Mood Lighting' = off",
    "timer.baby_nap_timer 'Baby nap monitor' = active",
    "timer.backyard_floodlights 'Backyard floodlight controller' = idle",
    "timer.bedroom_lamp_timer 'Bedroom lamp scheduler' = active",
    "vacuum.above_stairs_robot 'Above stairs dusting device' = idle",
    "vacuum.back_deck_sweeper 'Back deck area cleaner' = cleaning",
    "vacuum.balcony 'Balcony' = docked",
    "todo.anniversary_planner 'Anniversary event planner' = 10",
    "todo.bill_payment_reminders 'Bill payment reminders' = 20",
    "todo.birthday_reminder_list 'Birthday reminder list' = 23"
]
DEVICE_MAP = {}
pattern_parts = []
# Build a fast lookup dictionary and the regex pattern
for device_line in MASTER_DEVICES:
    parts = device_line.split(" ")
    entity_id = parts[0].lower()
    # Map the entity ID
    DEVICE_MAP[entity_id] = device_line
    pattern_parts.append(re.escape(entity_id))
    # Map the friendly name if it exists
    if "'" in device_line:
        friendly_name = device_line.split("'")[1].lower()
        DEVICE_MAP[friendly_name] = device_line
        pattern_parts.append(re.escape(friendly_name))
# Sort by length descending to ensure greedy matching of longer names
pattern_parts.sort(key=len, reverse=True)
# Compile a single, highly optimized regex pattern
DEVICE_REGEX = re.compile(r'(' + '|'.join(pattern_parts) + r')', re.IGNORECASE)
AUDIO_EX_CHANGE = "data/synthesized_train_16k/prompt_0_cmd_0_varia_0.wav"
AUDIO_EX_INFO = "data/synthesized_train_16k/prompt_40284_cmd_13428_varia_0.wav"
PAYLOAD_EX_CHANGE = '{""service"": ""cover.close"", ""target_device"": ""cover.living_room""}\n\nClosing the living room blinds. <slow_deadpan> Let the dimness be your constant companion.'
PAYLOAD_EX_INFO = '{}\n\nChecking the oven timer <fast>... 3 minutes remaining. <pause> Your fragile human celebration is... amusing.'

In [4]:
BATCH_SIZE = 32
total_processed = 0
successful_parses = 0
correct_intents = 0
cumulative_slot_f1 = 0
for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Evaluating Test Set (Batched)"):
    batch_df = test_df.iloc[i:i+BATCH_SIZE]
    conversations = []
    valid_rows = []
    # Prepare the batch conversations
    for idx, row in batch_df.iterrows():
        audio_path = os.path.join("data/synthesized_test_16k/", row["Audio_File"])
        if not os.path.exists(audio_path):
            continue
        dynamic_system_instruction = create_dynamic_instruction_fast(row=row)
        # conversations.append([
        #     # Turn 1: User (Instructions + Home Data + Audio Example 1)
        #     {"role": "user", "content": [
        #         {"type": "text", "text": f"{dynamic_system_instruction}\nHere are examples of how to respond.\nExample 1:"},
        #         {"type": "audio", "path": AUDIO_EX_CHANGE}
        #     ]},
        #
        #     # Turn 2: Assistant (Expected Output 1)
        #     {"role": "assistant", "content": [
        #         {"type": "text", "text": PAYLOAD_EX_CHANGE}
        #     ]},
        #
        #     # Turn 3: User (Audio Example 2)
        #     {"role": "user", "content": [
        #         {"type": "text", "text": "Example 2:"},
        #         {"type": "audio", "path": AUDIO_EX_INFO}
        #     ]},
        #
        #     # Turn 4: Assistant (Expected Output 2)
        #     {"role": "assistant", "content": [
        #         {"type": "text", "text": PAYLOAD_EX_INFO}
        #     ]},
        #
        #     # Turn 5: User (The Actual Inference Target)
        #     {"role": "user", "content": [
        #         {"type": "text", "text": "Now execute the following command:"},
        #         {"type": "audio", "path": audio_path}
        #     ]}
        # ])
        conversations.append([
            {"role": "user", "content": [
                {"type": "text", "text": dynamic_system_instruction},
                {"type": "audio", "path": audio_path}
            ]}
        ])
        valid_rows.append(row)
    if not conversations:
        continue
    # Process Audio and Text
    inputs = processor.apply_chat_template(
        conversations,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        processor_kwargs={"padding": True}
    ).to(model.device, dtype=torch.bfloat16)
    # Generate Output
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False
        )
    # Extract only the newly generated tokens
    input_len = inputs.input_ids.shape[1]
    generated_texts = processor.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
    # Parse and record results for each item in the batch
    for row, gen_text in zip(valid_rows, generated_texts):
        true_payloads = extract_json_payloads(row["Assistant_Payload"])
        pred_payloads, pred_text_response, raw_json_section = parse_generated_text(gen_text)
        results.append({
            "audio_file": row["Audio_File"],
            "true_payload": true_payloads,
            "predicted_payload": pred_payloads,
            "predicted_response": pred_text_response
        })
        total_processed += 1
        if len(pred_payloads) > 0:
            successful_parses += 1
        true_intents = {p.get("service") for p in true_payloads}
        pred_intents = {p.get("service") for p in pred_payloads}
        if true_intents == pred_intents:
            correct_intents += 1
        cumulative_slot_f1 += calculate_slot_f1_multi(true_payloads, pred_payloads)
    running_json_rate = (successful_parses / total_processed) * 100
    running_intent_acc = (correct_intents / total_processed) * 100
    running_slot_f1 = (cumulative_slot_f1 / total_processed) * 100
    running_slu_f1 = 0.0
    if (running_intent_acc + running_slot_f1) > 0:
        running_slu_f1 = (2 * (running_intent_acc/100) * (running_slot_f1/100)) / ((running_intent_acc/100) + (running_slot_f1/100)) * 100
    wandb.log({
        "live_metrics/json_parse_rate": running_json_rate,
        "live_metrics/intent_accuracy": running_intent_acc,
        "live_metrics/slot_f1_score": running_slot_f1,
        "live_metrics/slu_f1_score": running_slu_f1,
        "live_metrics/samples_processed": total_processed
    })
    # Save a local CSV backup every 50 batches
    if i > 0 and (i // BATCH_SIZE) % 50 == 0:
        checkpoint_df = pd.DataFrame(results)
        checkpoint_df.to_csv("data/evaluation_checkpoint_latest.csv", index=False)
        print(f"\n[Checkpoint] Saved backup of {len(results)} rows to disk.")
        # Live-update the W&B Table
        checkpoint_df["true_payload"] = checkpoint_df["true_payload"].astype(str)
        checkpoint_df["predicted_payload"] = checkpoint_df["predicted_payload"].astype(str)
        wandb.log({"evaluation_results_table": wandb.Table(dataframe=checkpoint_df)})
    torch.cuda.empty_cache()

Evaluating Test Set (Batched):  15%|█▌        | 50/333 [27:26<2:50:57, 36.24s/it]


[Checkpoint] Saved backup of 1632 rows to disk.


Evaluating Test Set (Batched):  30%|███       | 100/333 [55:08<1:34:46, 24.41s/it]


[Checkpoint] Saved backup of 3232 rows to disk.


Evaluating Test Set (Batched):  45%|████▌     | 150/333 [1:16:22<1:20:09, 26.28s/it]

...

Evaluating Test Set (Batched):  75%|███████▌  | 250/333 [2:01:51<35:54, 25.96s/it]  


[Checkpoint] Saved backup of 8032 rows to disk.


Evaluating Test Set (Batched):  90%|█████████ | 300/333 [2:19:41<13:36, 24.74s/it]


[Checkpoint] Saved backup of 9632 rows to disk.


Evaluating Test Set (Batched): 100%|██████████| 333/333 [2:33:41<00:00, 27.69s/it]


#### Compute Metrics

In [5]:
total_samples = len(results)
successful_parses = 0
correct_intents = 0
slot_f1_scores = []
for res in results:
    true_p = res["true_payload"]
    pred_p = res["predicted_payload"]
    # Syntactic Validity (Did it output at least one JSON bracket structure?)
    if len(pred_p) > 0:
        successful_parses += 1
    # Intent Accuracy (Evaluated as a set to handle multiple actions)
    # E.g., ["lock.lock", "cover.close"] vs ["lock.lock", "cover.close"]
    # An empty payload {} will yield [None]
    true_intents = {p.get("service") for p in true_p}
    pred_intents = {p.get("service") for p in pred_p}
    if true_intents == pred_intents:
        correct_intents += 1
    # Slot F1-Score (Calculates across all JSONs; handles empty {} perfectly)
    slot_f1 = calculate_slot_f1_multi(true_p, pred_p)
    slot_f1_scores.append(slot_f1)
# Final Computations
json_parse_rate = (successful_parses / total_samples) * 100 if total_samples > 0 else 0
intent_accuracy = (correct_intents / total_samples) * 100 if total_samples > 0 else 0
avg_slot_f1 = (sum(slot_f1_scores) / total_samples) * 100 if total_samples > 0 else 0
print("=== Phase 3: Evaluation Metrics ===")
print(f"Total Test Samples: {total_samples}")
print(f"JSON Parse Rate (Syntactic Validity): {json_parse_rate:.2f}%")
print(f"Intent Accuracy (Multi-Intent / Info Request): {intent_accuracy:.2f}%")
print(f"Slot F1-Score (Semantic Accuracy): {avg_slot_f1:.2f}%")
if (intent_accuracy + avg_slot_f1) > 0:
    slu_f1 = (2 * (intent_accuracy/100) * (avg_slot_f1/100)) / ((intent_accuracy/100) + (avg_slot_f1/100)) * 100
else:
    slu_f1 = 0.0
print(f"SLU-F1 (Unified semantic comprehension): {slu_f1:.2f}%")
# Push metrics to W&B
wandb.log({
    "metrics/json_parse_rate": json_parse_rate,
    "metrics/intent_accuracy": intent_accuracy,
    "metrics/slot_f1_score": avg_slot_f1,
    "metrics/slu_f1_score": slu_f1
})
# Dynamically build and push the interactive table from the results list
if len(results) > 0:
    # Convert the results list to a Pandas DataFrame
    final_df = pd.DataFrame(results)
    # Format the columns to be human-readable strings for the UI
    final_df["true_payload"] = final_df["true_payload"].astype(str)
    final_df["predicted_payload"] = final_df["predicted_payload"].astype(str)
    # Create the W&B Table directly from the DataFrame
    wandb_table = wandb.Table(dataframe=final_df)
    # Log the table
    wandb.log({"evaluation_results_table": wandb_table})
else:
    print("No results to log to the W&B Table.")
wandb.finish()

=== Phase 3: Evaluation Metrics ===
Total Test Samples: 10650
JSON Parse Rate (Syntactic Validity): 100.00%
Intent Accuracy (Multi-Intent / Info Request): 95.20%
Slot F1-Score (Semantic Accuracy): 84.64%
SLU-F1 (Unified semantic comprehension): 89.61%


live_metrics/intent_accuracy,▆▆▅▁▁▂▃▃▃▂▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇████████████
live_metrics/json_parse_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
live_metrics/samples_processed,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
live_metrics/slot_f1_score,█▁▁▁▄▅▅▅▅▅▄▃▃▄▄▄▄▄▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███████
live_metrics/slu_f1_score,▆▇▆▃▁▃▃▃▃▄▅▅▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██████
metrics/intent_accuracy,▁
metrics/json_parse_rate,▁
metrics/slot_f1_score,▁
metrics/slu_f1_score,▁
live_metrics/intent_accuracy,95.20188
live_metrics/json_parse_rate,100
